### 1. Setup and imports
 

In [3]:
import sys
import requests
import pandas
import dotenv
import matplotlib
import IPython
import json
import os
from dotenv import load_dotenv
from datetime import datetime

### 2. Load API key from .env

In [4]:
# Load environment variables
load_dotenv()

# Get API key
API_KEY = os.getenv('OPENWEATHER_API_KEY')

# Verify API key is loaded
if API_KEY:
    print(f"API Key loaded successfully: {API_KEY[:5]}...{API_KEY[-5:]}")
else:
    print("ERROR: API Key not found. Please check your .env file")

API Key loaded successfully: e3b97...f6185


### 3. Define cities to extract

In [5]:
# List of cities to get weather data for
CITIES = ["London", "New York", "Tokyo", "Sydney", "Cape Town", "Mumbai"]

print(f"Target cities: {len(CITIES)}")
for i, city in enumerate(CITIES, 1):
    print(f"  {i}. {city}")

Target cities: 6
  1. London
  2. New York
  3. Tokyo
  4. Sydney
  5. Cape Town
  6. Mumbai


### 4. Create extraction function

In [6]:
def fetch_weather(city_name):
    """
    Fetch current weather from OpenWeatherMap API
    
    Args:
        city_name: Name of the city
    
    Returns:
        dict: Weather data or None if failed
    """
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={API_KEY}&units=metric"
    
    print(f"Fetching: {city_name}")
    
    try:
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            print(f"   Success (Status: {response.status_code})")
            return response.json()
        else:
            print(f"   Failed (Status: {response.status_code})")
            print(f"   Error message: {response.json().get('message', 'Unknown error')}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"   Error: {e}")
        return None

### 5. Extract data for all cities

In [7]:
# Dictionary to store all raw data
raw_data = {}

print("Starting extraction...")
print("-" * 40)

for city in CITIES:
    raw_data[city] = fetch_weather(city)

print("-" * 40)
print("Extraction complete!")
print(f"Successful: {sum(1 for v in raw_data.values() if v is not None)}/{len(CITIES)} cities")

# List failed cities
failed = [city for city, data in raw_data.items() if data is None]
if failed:
    print(f"Failed cities: {', '.join(failed)}")

Starting extraction...
----------------------------------------
Fetching: London
   Success (Status: 200)
Fetching: New York
   Success (Status: 200)
Fetching: Tokyo
   Success (Status: 200)
Fetching: Sydney
   Success (Status: 200)
Fetching: Cape Town
   Success (Status: 200)
Fetching: Mumbai
   Success (Status: 200)
----------------------------------------
Extraction complete!
Successful: 6/6 cities


### 6. Preview raw data structure

In [8]:
# Look at raw structure for first successful city
for city, data in raw_data.items():
    if data:
        print(f"Sample data for {city}:")
        print(f"  Temperature: {data['main']['temp']}C")
        print(f"  Feels like: {data['main']['feels_like']}C")
        print(f"  Humidity: {data['main']['humidity']}%")
        print(f"  Pressure: {data['main']['pressure']} hPa")
        print(f"  Conditions: {data['weather'][0]['description']}")
        print(f"  Wind speed: {data['wind']['speed']} m/s")
        print(f"  Clouds: {data['clouds']['all']}%")
        print(f"  Country: {data['sys']['country']}")
        break

Sample data for London:
  Temperature: 15.87C
  Feels like: 14.45C
  Humidity: 36%
  Pressure: 1024 hPa
  Conditions: overcast clouds
  Wind speed: 2.68 m/s
  Clouds: 88%
  Country: GB


### 7. Display all extracted data summary

In [9]:
# Print summary for all successful cities
print("Extracted Data Summary:")
print("=" * 50)

for city, data in raw_data.items():
    if data:
        temp = data['main']['temp']
        humidity = data['main']['humidity']
        conditions = data['weather'][0]['description']
        print(f"{city:15} | {temp:5.1f}C | {humidity:3d}% | {conditions}")
    else:
        print(f"{city:15} | FAILED TO EXTRACT")

Extracted Data Summary:
London          |  15.9C |  36% | overcast clouds
New York        |  11.7C |  72% | mist
Tokyo           |  17.5C |  77% | broken clouds
Sydney          |  12.4C |  78% | broken clouds
Cape Town       |  14.2C |  74% | overcast clouds
Mumbai          |  30.0C |  66% | haze


### 8. Create data directory and save raw data

In [10]:
# Create data directory if it doesn't exist
os.makedirs('../data', exist_ok=True)

# Save raw data to JSON file
with open('../data/raw_extract.json', 'w') as f:
    json.dump(raw_data, f, indent=2)

print("Raw data saved to: data/raw_extract.json")

# Check file size
file_size = os.path.getsize('../data/raw_extract.json')
print(f"File size: {file_size} bytes")

# Also save with timestamp for historical tracking
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_file = f'../data/raw_extract_{timestamp}.json'
with open(backup_file, 'w') as f:
    json.dump(raw_data, f, indent=2)
print(f"Backup saved to: {backup_file}")

Raw data saved to: data/raw_extract.json
File size: 5422 bytes
Backup saved to: ../data/raw_extract_20260419_165629.json


### 9. Save to CSV as well (optional)

In [11]:
# Convert to DataFrame and save as CSV
import pandas as pd

# Prepare data for DataFrame
rows = []
for city, data in raw_data.items():
    if data:
        rows.append({
            'city': city,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'temperature_c': data['main']['temp'],
            'feels_like_c': data['main']['feels_like'],
            'humidity_pct': data['main']['humidity'],
            'pressure_hpa': data['main']['pressure'],
            'weather_main': data['weather'][0]['main'],
            'weather_description': data['weather'][0]['description'],
            'wind_speed_ms': data['wind']['speed'],
            'clouds_pct': data['clouds']['all'],
            'country': data['sys']['country']
        })

if rows:
    df = pd.DataFrame(rows)
    df.to_csv('../data/raw_extract.csv', index=False)
    print(f"CSV saved to: data/raw_extract.csv")
    print(f"Records saved: {len(df)}")
else:
    print("No data to save to CSV")

CSV saved to: data/raw_extract.csv
Records saved: 6


### 10. Verify extraction completion

In [12]:
# Final verification
print("Extraction Stage Complete!")
print("=" * 40)
print(f"Total cities processed: {len(CITIES)}")
print(f"Successful extractions: {sum(1 for v in raw_data.values() if v is not None)}")
print(f"Failed extractions: {sum(1 for v in raw_data.values() if v is None)}")
print("\nOutput files created:")
print("  - data/raw_extract.json")
print("  - data/raw_extract.csv")
print("  - data/raw_extract_[timestamp].json")
print("\nReady for next stage: 2_transform.ipynb")

Extraction Stage Complete!
Total cities processed: 6
Successful extractions: 6
Failed extractions: 0

Output files created:
  - data/raw_extract.json
  - data/raw_extract.csv
  - data/raw_extract_[timestamp].json

Ready for next stage: 2_transform.ipynb
